<a href="https://colab.research.google.com/github/nmach22/movie-sentiment-analysis/blob/main/notebooks/colab_biGRU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab environment

### Imports and installs

In [1]:
import pandas as pd
import numpy as np

import torch

import zipfile
import os

In [2]:
!pip install kaggle -q
!pip install tensorboard -q

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Set up Kaggle API credentials

To download datasets from Kaggle, you need an API key. Follow these steps:

1. Go to [Kaggle](https://www.kaggle.com/).
2. Log in to your account.
3. Click on your profile picture in the top right corner and select "My Account".
4. Scroll down to the "API" section and click "Create New API Token". This will download a `kaggle.json` file.
5. Upload this `kaggle.json` file to your Colab environment using the file upload button on the left sidebar (folder icon) or by running the following Python code to open an upload dialog.

In [4]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"nmach22","key":"1bcaaf4b5ace614578f4d815af992b7e"}'}

In [5]:
# Create .kaggle directory if it doesn't exist
!mkdir -p ~/.kaggle

# Move kaggle.json to the .kaggle directory
!mv kaggle.json ~/.kaggle/

# Set permissions for the kaggle.json file
!chmod 600 ~/.kaggle/kaggle.json

### Download competition

In [6]:
import kaggle

competition_name = 'sentiment-analysis-on-movie-reviews'

print(f"Downloading files for competition: {competition_name}")
kaggle.api.competition_download_files(competition_name, path='./')
print("Download complete.")

Download complete.


### Unzip recursively

In [7]:
def unzip_recursive(file_path, extract_to='./'):
    """
    Unzips a specific file and recursively unzips any zip files
    found within its extracted contents.
    """
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        return

    print(f"--- Extracting: {file_path} ---")

    try:
        with zipfile.ZipFile(file_path, 'r') as zip_ref:
            # Get list of filenames before extracting to track them
            extracted_members = zip_ref.namelist()
            zip_ref.extractall(extract_to)

        os.remove(file_path)

        # Look specifically at what we just extracted
        for member in extracted_members:
            member_path = os.path.join(extract_to, member)

            # If an extracted file is another zip, recurse into it
            if member_path.endswith('.zip'):
                unzip_recursive(member_path, extract_to)

    except zipfile.BadZipFile:
        print(f"Error: {file_path} is corrupt or not a valid zip.")

# Start the process with your specific main file
main_file = './sentiment-analysis-on-movie-reviews.zip'
unzip_recursive(main_file)

print("\nProcessing complete. Final directory state:")
!ls -F

--- Extracting: ./sentiment-analysis-on-movie-reviews.zip ---
--- Extracting: ./test.tsv.zip ---
--- Extracting: ./train.tsv.zip ---

Processing complete. Final directory state:
drive/	sample_data/  sampleSubmission.csv  test.tsv  train.tsv


# Split and preprocess

## Split

In [8]:
train_df = pd.read_csv('train.tsv', sep='\t')

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(train_df['Phrase'], train_df['Sentiment'], test_size=0.2, random_state=42)

print(f"Train phrases: {X_train.shape[0]} samples, shape: {X_train.shape}")
print(f"Validation phrases: {X_val.shape[0]} samples, shape: {X_val.shape}")
print(f"Train sentiments: {y_train.shape[0]} samples, shape: {y_train.shape}")
print(f"Validation sentiments: {y_val.shape[0]} samples, shape: {y_val.shape}")

Train phrases: 124848 samples, shape: (124848,)
Validation phrases: 31212 samples, shape: (31212,)
Train sentiments: 124848 samples, shape: (124848,)
Validation sentiments: 31212 samples, shape: (31212,)


In [10]:
X_train.head()

,Phrase
22538,cheesy
99237,unintentionally -RRB-
60377,will need all the luck they can muster just fi...
128317,somewhere between Sling Blade and South of Hea...
20776,reminds at every turn of Elizabeth Berkley 's ...


## Text Preprocessing & Tokenization

In [11]:
import re

def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text.split()

X_train_tok = X_train.apply(tokenize)
X_val_tok   = X_val.apply(tokenize)

train_mask = X_train_tok.apply(len) > 0
val_mask   = X_val_tok.apply(len)   > 0

X_train_tok = X_train_tok[train_mask]
y_train     = y_train[train_mask]

X_val_tok   = X_val_tok[val_mask]
y_val       = y_val[val_mask]

print(f"Dropped {(~train_mask).sum()} empty train samples")
print(f"Dropped {(~val_mask).sum()} empty val samples")

Dropped 21 empty train samples
Dropped 6 empty val samples


### build vocabulary

In [12]:
from collections import Counter

PAD_IDX, UNK_IDX = 0, 1

counter = Counter(token for tokens in X_train_tok for token in tokens)

# Drop rare words
vocab = {"<PAD>": PAD_IDX, "<UNK>": UNK_IDX}
for word, freq in counter.items():
    if freq >= 2:
        vocab[word] = len(vocab)

VOCAB_SIZE = len(vocab)

### Numericalize & Pad Sequences

In [13]:
import torch
from torch.nn.utils.rnn import pad_sequence

def encode(tokens):
    return [vocab.get(t, UNK_IDX) for t in tokens]

def to_tensor_and_lengths(encoded_series):
    encoded_list = [encode(t) for t in encoded_series]
    lengths = torch.tensor([len(t) for t in encoded_list], dtype=torch.long)
    tensors = [torch.tensor(t, dtype=torch.long) for t in encoded_list]
    padded_tensors = pad_sequence(tensors, batch_first=True, padding_value=PAD_IDX)
    return padded_tensors, lengths

X_train_tensor, X_train_lengths = to_tensor_and_lengths(X_train_tok)
X_val_tensor, X_val_lengths     = to_tensor_and_lengths(X_val_tok)
y_train_tensor                  = torch.tensor(y_train.values, dtype=torch.long)
y_val_tensor                    = torch.tensor(y_val.values,   dtype=torch.long)

## Chackpoints

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)  # should print: cuda

cuda


In [15]:
def save_checkpoint(model, optimizer, scheduler, epoch, val_acc, path="checkpoints"):
    os.makedirs(path, exist_ok=True)
    torch.save({
        "epoch":            epoch,
        "model_state":      model.state_dict(),
        "optimizer_state":  optimizer.state_dict(),
        "scheduler_state":  scheduler.state_dict(),
        "val_acc":          val_acc,
    }, f"{path}/epoch_{epoch}_acc{val_acc:.4f}.pt")

def load_checkpoint(path, model, optimizer=None, scheduler=None, map_location=device):
    checkpoint = torch.load(path, map_location=map_location)

    # ── Detect format ──
    if "model_state" in checkpoint:
        # Full checkpoint saved by save_checkpoint()
        model.load_state_dict(checkpoint["model_state"])
        if optimizer and "optimizer_state" in checkpoint:
            optimizer.load_state_dict(checkpoint["optimizer_state"])
        if scheduler and "scheduler_state" in checkpoint:
            scheduler.load_state_dict(checkpoint["scheduler_state"])
        epoch    = checkpoint.get("epoch", 0)
        val_acc  = checkpoint.get("val_acc", 0.0)
    else:
        # Raw state dict — weights only, no training state
        model.load_state_dict(checkpoint)
        epoch, val_acc = 0, 0.0
        print("Warning: raw weights loaded — optimizer/scheduler not restored.")

    print(f"Restored → Epoch {epoch} | Val Acc {val_acc:.4f}")
    return epoch, val_acc

## Create Dataset & DataLoader

In [16]:
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 8192

train_loader = DataLoader(
    TensorDataset(X_train_tensor, X_train_lengths, y_train_tensor),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,        # parallel data loading
    pin_memory=True       # faster CPU→GPU transfer
    )

val_loader = DataLoader(
    TensorDataset(X_val_tensor, X_val_lengths, y_val_tensor),
    batch_size=BATCH_SIZE,
    num_workers=2,
    pin_memory=True
    )

## Define the LSTM Model

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class BiGRUSentiment(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256,
                 output_dim=5, n_layers=2, dropout=0.3, kernel_size=3):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)

        # CNN to catch n-grams (local context)
        self.conv = nn.Conv1d(in_channels=embed_dim, out_channels=embed_dim,
                              kernel_size=kernel_size, padding=kernel_size // 2)

        # GRU Layer: Faster and often more efficient than LSTM
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout if n_layers > 1 else 0,
                          bidirectional=True)

        # Attention Layer
        self.attention = nn.Linear(hidden_dim * 2, 1)

        # Deep Head
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, lengths):
        # 1. Embedding
        embedded = self.dropout(self.embedding(x))

        # 2. CNN for local feature extraction
        conved = embedded.permute(0, 2, 1)
        conved = F.relu(self.conv(conved))
        conved = conved.permute(0, 2, 1)

        # 3. Packed GRU
        packed_input = pack_padded_sequence(conved, lengths.cpu(),
                                            batch_first=True, enforce_sorted=False)
        # Note: GRU returns (output, hidden) - no 'cell state' like LSTM
        packed_output, _ = self.gru(packed_input)
        output, _ = pad_packed_sequence(packed_output, batch_first=True)

        # 4. Attention
        energy = torch.tanh(self.attention(output))
        weights = F.softmax(energy, dim=1)
        context = torch.sum(output * weights, dim=1)

        return self.fc(context)

In [18]:
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None: nn.init.constant_(m.bias, 0)

    elif isinstance(m, nn.Conv1d):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')

    elif isinstance(m, nn.GRU):
        for name, param in m.named_parameters():
            if 'weight_ih' in name:
                nn.init.orthogonal_(param.data)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param.data)
            elif 'bias' in name:
                nn.init.constant_(param.data, 0)

    elif isinstance(m, nn.Embedding):
        nn.init.uniform_(m.weight, -0.1, 0.1)

In [58]:
import torch.optim as optim

model = BiGRUSentiment(vocab_size=VOCAB_SIZE, embed_dim=32,
                             hidden_dim=64, output_dim=5,
                             n_layers=7, dropout=0.3).to(device)

# Apply weights
model.apply(init_weights)

# Ensure the padding index stays zeroed out
model.embedding.weight.data[PAD_IDX].zero_()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

In [59]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params}")

Total number of parameters: 1018214


### Train the Model

In [60]:
EPOCHS       = 30
best_val_acc = 0.0
patience     = 0
MAX_PATIENCE = 5   # early stopping

CHECKPOINT_DIR = "/content/drive/MyDrive/movie_sentiment_checkpoints"

In [66]:
def train_epoch(model, loader):
    model.train()
    total_loss, correct = 0, 0
    for X_batch, len_batch, y_batch in loader:
        X_batch   = X_batch.to(device)
        len_batch = len_batch.to(device)
        y_batch   = y_batch.to(device)

        optimizer.zero_grad()
        preds = model(X_batch, len_batch)
        loss  = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct    += (preds.argmax(1) == y_batch).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

def eval_epoch(model, loader):
    model.eval()
    total_loss, correct = 0, 0
    with torch.no_grad():
        for X_batch, len_batch, y_batch in loader:
            X_batch   = X_batch.to(device)
            len_batch = len_batch.to(device)
            y_batch   = y_batch.to(device)

            preds = model(X_batch, len_batch)
            total_loss += criterion(preds, y_batch).item()
            correct    += (preds.argmax(1) == y_batch).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)



for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader)
    vl_loss, vl_acc = eval_epoch(model, val_loader)
    scheduler.step(vl_acc)

    print(f"Epoch {epoch:02d} | Train {tr_acc:.4f} | Val {vl_acc:.4f}")

    # ── Save every epoch (optional, comment out if disk is tight) ──
    save_checkpoint(model, optimizer, scheduler, epoch, vl_acc)

    # ── Save best separately, always overwrite ──
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        patience     = 0
        torch.save(model.state_dict(), f"{CHECKPOINT_DIR}/{model.__class__.__name__}.pt")
        print(f"  ✓ New best saved ({best_val_acc:.4f})")
    else:
        patience += 1
        print(f"  No improvement ({patience}/{MAX_PATIENCE})")
        if patience >= MAX_PATIENCE:
            print("Early stopping triggered.")
            break


Epoch 01 | Train 0.7014 | Val 0.6380
  ✓ New best saved (0.6380)
Epoch 02 | Train 0.7026 | Val 0.6394
  ✓ New best saved (0.6394)
Epoch 03 | Train 0.7093 | Val 0.6398
  ✓ New best saved (0.6398)
Epoch 04 | Train 0.7118 | Val 0.6324
  No improvement (1/5)
Epoch 05 | Train 0.7141 | Val 0.6410
  ✓ New best saved (0.6410)
Epoch 06 | Train 0.7178 | Val 0.6368
  No improvement (1/5)
Epoch 07 | Train 0.7199 | Val 0.6440
  ✓ New best saved (0.6440)
Epoch 08 | Train 0.7228 | Val 0.6444
  ✓ New best saved (0.6444)
Epoch 09 | Train 0.7257 | Val 0.6433
  No improvement (1/5)
Epoch 10 | Train 0.7270 | Val 0.6421
  No improvement (2/5)
Epoch 11 | Train 0.7298 | Val 0.6453
  ✓ New best saved (0.6453)
Epoch 12 | Train 0.7302 | Val 0.6414
  No improvement (1/5)
Epoch 13 | Train 0.7335 | Val 0.6502
  ✓ New best saved (0.6502)
Epoch 14 | Train 0.7348 | Val 0.6391
  No improvement (1/5)
Epoch 15 | Train 0.7371 | Val 0.6511
  ✓ New best saved (0.6511)
Epoch 16 | Train 0.7377 | Val 0.6480
  No improvement (

In [67]:
model.eval()
with torch.no_grad():
    # 1. Get the slice of data
    x_samples = X_val_tensor[:100].to(device)

    # 2. Calculate lengths
    # This counts non-zero elements per row
    lengths = (x_samples != PAD_IDX).sum(dim=1).cpu()

    # 3. Pass both to the model
    logits = model(x_samples, lengths)

    sample_preds = logits.argmax(1)

    print(f"Predictions: {sample_preds}")
    print(f"Unique classes predicted: {sample_preds.unique()}")

Predictions: tensor([2, 3, 2, 2, 1, 1, 3, 3, 2, 2, 0, 1, 0, 3, 1, 3, 2, 2, 2, 2, 2, 1, 2, 2,
        2, 2, 2, 1, 2, 2, 2, 1, 2, 2, 4, 2, 2, 4, 2, 4, 3, 2, 2, 2, 2, 4, 2, 2,
        1, 2, 4, 2, 2, 3, 1, 2, 3, 2, 1, 2, 2, 4, 2, 0, 1, 2, 1, 2, 2, 3, 2, 1,
        2, 2, 4, 2, 2, 2, 3, 1, 1, 2, 2, 2, 2, 2, 0, 2, 3, 2, 2, 3, 2, 3, 3, 2,
        4, 3, 0, 2], device='cuda:0')
Unique classes predicted: tensor([0, 1, 2, 3, 4], device='cuda:0')


# Generate Predictions for Submission

In [68]:
test_df = pd.read_csv("test.tsv", sep="\t")

# Fill NaN values in 'Phrase' with empty string before tokenization
test_df['Phrase'] = test_df['Phrase'].fillna('')

X_test_tok = test_df['Phrase'].apply(tokenize)

# to_tensor_and_lengths now returns both tensor and lengths
X_test_tensor, X_test_lengths = to_tensor_and_lengths(X_test_tok)

In [69]:
from torch.utils.data import DataLoader, TensorDataset

# Create a simple dataset and loader
test_dataset = TensorDataset(X_test_tensor, X_test_lengths)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

all_preds = []

model.eval()
with torch.no_grad():
    for x_batch, len_batch in test_loader:
        # Clamp here just in case
        len_batch = len_batch.clamp(min=1)
        logits = model(x_batch.to(device), len_batch)
        all_preds.append(logits.argmax(1).cpu())

preds = torch.cat(all_preds).numpy()

In [70]:
submission = pd.DataFrame({"PhraseId": test_df["PhraseId"], "Sentiment": preds})
submission.to_csv(f"{model.__class__.__name__}_submission.csv", index=False)